# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook demonstrates how to explore, process, and analyze the FAIR² Mental Health Survey dataset from Kilifi County, Kenya using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL for the dataset
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata and create Dataset instance
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Published: {getattr(metadata, 'datePublished', 'N/A')}")
print(f"Licensing: {getattr(metadata, 'license', 'N/A')}")

## 2. Data Overview
Let's examine the available record sets, their fields (columns), and corresponding `@id`s.

We inspect the main record sets and their fields by their `@id`.

In [ ]:
# List all available record sets and their IDs
record_sets = dataset.list_record_sets()
print("Available record sets (by @id):")
for rset in record_sets:
    print(f" - {rset['@id']}: {rset.get('name', '')}")

# For each record set, list its fields and columns, referencing their @id

print("\nRecord sets and their fields (columns):")
for rset in record_sets:
    print(f"\nRecord Set: {rset.get('name','')} [{rset['@id']}]")
    fields = dataset.list_fields(rset['@id'])
    for field in fields:
        print(f"  - Field: {field.get('name','')} [@id={field['@id']}] \tColumn: {field.get('column', '')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

We demonstrate how to load data from all record sets present in the schema.

In [ ]:
# Identify the record sets we want to load (by @id)
# For this dataset, let's assume main survey records are in 'mlc-recordset-main-survey'
# If you explored the record sets above, replace below with the actual @ids from the dataset.

main_record_set_id = None

# Try to automatically pick a main, relevant record set (one with 'survey', 'main', or 'records' in name, or pick first)
for rset in dataset.list_record_sets():
    if main_record_set_id:
        break
    rset_name = rset.get('name','').lower()
    if any(key in rset_name for key in ['survey', 'main', 'records']):
        main_record_set_id = rset['@id']
if not main_record_set_id and len(dataset.list_record_sets())>0:
    main_record_set_id = dataset.list_record_sets()[0]['@id']

# For demonstration, collect all record_set @ids
record_sets_ids = [rset['@id'] for rset in dataset.list_record_sets()]

dataframes = {}
for rset_id in record_sets_ids:
    # Load records as DataFrame
    records = list(dataset.records(record_set=rset_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rset_id] = df
        print(f"Loaded {df.shape[0]} records for record set '@id'={rset_id}")
    else:
        print(f"No records found for record set '@id'={rset_id}")

if main_record_set_id and main_record_set_id in dataframes:
    print("\nColumns in main record set:")
    print(dataframes[main_record_set_id].columns.tolist())
    print(dataframes[main_record_set_id].head())
else:
    print("No main survey DataFrame found.")

## 4. Exploratory Data Analysis (EDA)

Let's do common EDA steps: filtering out records with high GAD-7 scores, normalizing, and grouping by a demographic attribute.

NOTE: Replace `<numeric_field_id>` and `<group_field>` with appropriate `@id` or column names found in your previous overview. For this dataset, GAD-7 and age are good candidates.

In [ ]:
# Choose a numeric field (e.g., 'gad7_score') and grouping field (e.g., 'gender') from your columns above
# For demonstration: Search for likely field names

df = dataframes[main_record_set_id]
numeric_field_candidates = [col for col in df.columns if any(sub in col.lower() for sub in ['gad', 'phq', 'psq', 'score', 'age'])]
print("Numeric field candidates:", numeric_field_candidates)

group_field_candidates = [col for col in df.columns if any(sub in col.lower() for sub in ['gender', 'sex', 'education', 'village', 'marital'])]
print("Group field candidates:", group_field_candidates)

# Select a default (or adjust below)
numeric_field = None
for x in ['gad7_score', 'phq9_score', 'psq_score', 'age']:
    if x in df.columns:
        numeric_field = x
        break
if not numeric_field and numeric_field_candidates:
    numeric_field = numeric_field_candidates[0]

group_field = None
for x in ['gender', 'sex', 'education', 'village', 'marital']:
    if x in df.columns:
        group_field = x
        break
if not group_field and group_field_candidates:
    group_field = group_field_candidates[0] if group_field_candidates else None

print(f"Using numeric field: {numeric_field}")
print(f"Using group field: {group_field}")

# Filter records where numeric field > threshold (e.g., GAD-7 > 10 for moderate-to-severe anxiety)
threshold = 10
if numeric_field and numeric_field in df.columns:
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())
    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
    # Group
    if group_field and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"\nMean {numeric_field} by {group_field} for filtered sample:")
        print(grouped_df)
else:
    print("No suitable numeric field for EDA found.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Let's plot the numeric field distribution and the group mean as a bar chart.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field and numeric_field in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), bins=20)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()
    
    if group_field and group_field in df.columns:
        plt.figure(figsize=(8,5))
        sns.barplot(x=group_field, y=numeric_field, data=df, ci=None)
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.ylabel(f"Mean {numeric_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion

We have demonstrated how to use the `mlcroissant` library to:
- Load metadata and data from a FAIR-compliant Croissant dataset schema using only the dataset URL,
- Explore available record sets and fields via their `@id`,
- Load records directly as DataFrames for analysis,
- Perform basic filtering and normalization for key mental health indicators,
- Quickly visualize and summarize group-level patterns.

This approach ensures robust, reproducible, and FAIR analytics for AI-ready health data sourced from international standards.